In [1]:

from pathlib import Path

# Current working directory ko project root par normalize karo
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_DIR = CURRENT_DIR.parent
else:
    PROJECT_DIR = CURRENT_DIR

ARTIFACT_DIR = PROJECT_DIR / "models"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
RAW_DIR = PROJECT_DIR / "data" / "raw"

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = RAW_DIR / "zomato_cleaned.csv"

print("Project root:", PROJECT_DIR)
print("Raw data:", RAW_DIR)
print("Processed data:", PROCESSED_DIR)
print("Models:", ARTIFACT_DIR)
print("Dataset exists:", DATA_PATH.exists())


Project root: d:\Projects\LEARNINGS\MACHINE LEARNING\Swiftdrop delivery pipeline
Raw data: d:\Projects\LEARNINGS\MACHINE LEARNING\Swiftdrop delivery pipeline\data\raw
Processed data: d:\Projects\LEARNINGS\MACHINE LEARNING\Swiftdrop delivery pipeline\data\processed
Models: d:\Projects\LEARNINGS\MACHINE LEARNING\Swiftdrop delivery pipeline\models
Dataset exists: True


In [2]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd
import mlflow
import joblib

# When opened from the repository's notebooks/ directory, add the repo root.
REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data_loader import load_dataset
from src.preprocessing import TARGET, prepare_model_frame, make_preprocessor
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

raw = load_dataset(str(DATA_PATH) if DATA_PATH and DATA_PATH.exists() else None)
frame = prepare_model_frame(raw)

print("Raw rows:", len(raw))
print("Rows after target validation:", len(frame))
print("Target missing values:", int(frame[TARGET].isna().sum()))
print("Feature columns:", frame.drop(columns=[TARGET]).columns.tolist())
display(frame.head())

Raw rows: 38964
Rows after target validation: 38964
Target missing values: 0
Feature columns: ['Delivery_person_Age', 'Delivery_person_Ratings', 'Restaurant_latitude', 'Restaurant_longitude', 'Delivery_location_latitude', 'Delivery_location_longitude', 'Weather_conditions', 'Road_traffic_density', 'Vehicle_condition', 'Type_of_order', 'Type_of_vehicle', 'multiple_deliveries', 'Festival', 'City', 'distance_km', 'order_day_of_week', 'order_month', 'order_day', 'order_is_weekend', 'Order_Date_parse_failed', 'Time_Orderd_was_missing', 'Time_Orderd_parse_failed', 'order_time_minutes', 'order_time_sin', 'order_time_cos']


,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Weather_conditions,Road_traffic_density,Vehicle_condition,Type_of_order,...,order_day_of_week,order_month,order_day,order_is_weekend,Order_Date_parse_failed,Time_Orderd_was_missing,Time_Orderd_parse_failed,order_time_minutes,order_time_sin,order_time_cos
0,36.0,4.2,30.327968,78.046106,30.397968,78.116106,Fog,Jam,2,Snack,...,5.0,2.0,12.0,1.0,0,0,0,1315.0,-0.518773,0.854912
1,21.0,4.7,10.003064,76.307589,10.043064,76.347589,Stormy,High,1,Meal,...,6.0,2.0,13.0,1.0,0,0,0,895.0,-0.691513,-0.722364
2,23.0,4.7,18.562450,73.916619,18.652450,74.006619,Sandstorms,Medium,1,Drinks,...,4.0,3.0,4.0,0.0,0,0,0,1050.0,-0.991445,-0.130526
3,34.0,4.3,30.899584,75.809346,30.919584,75.829346,Sandstorms,Low,0,Buffet,...,6.0,2.0,13.0,1.0,0,0,0,560.0,0.642788,-0.766044
4,24.0,4.7,26.463504,80.372929,26.593504,80.502929,Fog,Jam,1,Snack,...,0.0,2.0,14.0,0.0,0,0,0,1190.0,-0.887011,0.461749


In [3]:
# Optional: save a no-null cleaned dataset for inspection/sharing only.
# This artifact is NOT used for model validation because its imputations use all rows.
from src.preprocessing import make_fully_clean_export

clean_export = make_fully_clean_export(frame)
clean_export_path = PROJECT_DIR / "data" / "processed" / "zomato_cleaned_feature_engineered.csv"
clean_export_path.parent.mkdir(parents=True, exist_ok=True)
clean_export.to_csv(clean_export_path, index=False)

assert clean_export.isna().sum().sum() == 0
print("Saved no-null inspection artifact:", clean_export_path)
print("Missing cells:", int(clean_export.isna().sum().sum()))

Saved no-null inspection artifact: d:\Projects\LEARNINGS\MACHINE LEARNING\Swiftdrop delivery pipeline\data\processed\zomato_cleaned_feature_engineered.csv
Missing cells: 0


In [4]:
X = frame.drop(columns=[TARGET])
y = frame[TARGET].astype(float)

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = XGBRegressor(
    n_estimators=700,
    max_depth=7,
    learning_rate=0.04,
    subsample=0.85,
    colsample_bytree=0.85,
    objective="reg:squarederror",
    eval_metric="mae",
    random_state=42,
    n_jobs=-1,
)

pipeline = Pipeline([
    ("preprocessor", make_preprocessor(X_train, scale_numeric=False)),
    ("model", model),
])

print("Train rows:", len(X_train), "| Validation rows:", len(X_valid))
print("Training feature nulls before imputation:", int(X_train.isna().sum().sum()))
print("Validation feature nulls before imputation:", int(X_valid.isna().sum().sum()))

Train rows: 31171 | Validation rows: 7793
Training feature nulls before imputation: 4548
Validation feature nulls before imputation: 1138


In [6]:
pickup_related = [
    col for col in frame.columns
    if "picked" in col.lower()
]

print("Pickup-related columns:", pickup_related)

Pickup-related columns: []


In [7]:
mlflow.set_experiment("SwiftDrop-XGBoost-Baseline")

with mlflow.start_run() as run:
    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_valid)

    mae = float(mean_absolute_error(y_valid, preds))
    rmse = float(np.sqrt(mean_squared_error(y_valid, preds)))
    r2 = float(r2_score(y_valid, preds))

    mlflow.log_param("dataset_source_url", "https://huggingface.co/datasets/allenborochin/zomato_delivery_EDA")
    mlflow.log_param("target", TARGET)
    mlflow.log_param("split_random_state", 42)
    mlflow.log_param("validation_size", 0.2)
    mlflow.log_param("n_estimators", 700)
    mlflow.log_param("max_depth", 7)
    mlflow.log_param("learning_rate", 0.04)
    mlflow.log_metric("validation_mae", mae)
    mlflow.log_metric("validation_rmse", rmse)
    mlflow.log_metric("validation_r2", r2)

    metrics = {
        "validation_mae": mae,
        "validation_rmse": rmse,
        "validation_r2": r2,
        "training_rows": len(y_train),
        "validation_rows": len(y_valid),
        "mlflow_run_id": run.info.run_id,
    }

    joblib.dump(pipeline, ARTIFACT_DIR / "xgboost_pipeline.joblib")
    (ARTIFACT_DIR / "xgboost_metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
    mlflow.log_artifact(str(ARTIFACT_DIR / "xgboost_metrics.json"))

print(json.dumps(metrics, indent=2))

{
  "validation_mae": 3.0842244143236988,
  "validation_rmse": 3.849758362690857,
  "validation_r2": 0.8258568483239679,
  "training_rows": 31171,
  "validation_rows": 7793,
  "mlflow_run_id": "2471bc7a75884cfc8f1d7e4a40cf7210"
}


In [8]:
# Baseline sanity checks
assert np.isfinite(preds).all()
assert mae > 0
assert len(preds) == len(y_valid)
print("Baseline completed. Use the measured validation MAE in later comparisons; do not invent it.")
print("Pipeline:", ARTIFACT_DIR / "xgboost_pipeline.joblib")
print("Metrics:", ARTIFACT_DIR / "xgboost_metrics.json")

Baseline completed. Use the measured validation MAE in later comparisons; do not invent it.
Pipeline: d:\Projects\LEARNINGS\MACHINE LEARNING\Swiftdrop delivery pipeline\models\xgboost_pipeline.joblib
Metrics: d:\Projects\LEARNINGS\MACHINE LEARNING\Swiftdrop delivery pipeline\models\xgboost_metrics.json


In [9]:
MODELS_DIR = PROJECT_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

In [10]:
import joblib

PIPELINE_PATH = MODELS_DIR / "xgboost_pipeline.joblib"

xgb_pipeline = joblib.load(PIPELINE_PATH)

print("Pipeline loaded:", type(xgb_pipeline))
print("Steps:", list(xgb_pipeline.named_steps.keys()))

Pipeline loaded: <class 'sklearn.pipeline.Pipeline'>
Steps: ['preprocessor', 'model']


In [12]:
import onnx
from onnxmltools import convert_xgboost
from onnxmltools.convert.common.data_types import FloatTensorType

# Extract the trained XGBoost estimator
xgb_model = xgb_pipeline.named_steps["model"]

# Get the number of features expected by the trained model
n_features = xgb_model.n_features_in_

# Define ONNX input shape
initial_types = [
    ("features", FloatTensorType([None, n_features]))
]

# Convert XGBoost model to ONNX
xgb_onnx = convert_xgboost(
    xgb_model,
    initial_types=initial_types,
    target_opset=15,
)

# Save ONNX model
XGB_ONNX_PATH = MODELS_DIR / "xgboost_model.onnx"
onnx.save_model(xgb_onnx, str(XGB_ONNX_PATH))

# Verify exported model
onnx.checker.check_model(str(XGB_ONNX_PATH))

print("Export successful!")
print("ONNX path:", XGB_ONNX_PATH)
print("Input features:", n_features)
print("File size:", round(XGB_ONNX_PATH.stat().st_size / 1024, 2), "KB")

Export successful!
ONNX path: d:\Projects\LEARNINGS\MACHINE LEARNING\Swiftdrop delivery pipeline\models\xgboost_model.onnx
Input features: 46
File size: 4893.74 KB
